In [1]:
import os
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from langchain_core.messages import BaseMessage

# ---------------------------------------------------------------------------
# 1. 环境变量与模型初始化
# ---------------------------------------------------------------------------

# 从 .env 文件中加载环境变量；override=True 表示同名变量以 .env 文件为准
load_dotenv(override=True)

# 从环境变量中读取 DeepSeek 的 API Key 和 API Base 地址
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_API_BASE = os.getenv("DEEPSEEK_API_BASE")

# 初始化聊天模型：
# model 采用 "provider:model_name" 的格式指定 DeepSeek 的模型
# api_key / base_url 用于覆盖默认配置，指向自定义的 DeepSeek 服务
model = init_chat_model(
      model="deepseek:deepseek-v4-flash",
      # model_provider="deepseek",  # 已在 model 字符串中隐含指定，可省略
      api_key=DEEPSEEK_API_KEY,
      base_url=DEEPSEEK_API_BASE,
)

## 举例

In [2]:
from langchain.tools import tool
from langchain.messages import HumanMessage


@tool(parse_docstring=True)
def get_weather(city: str) -> str:
      """
      获取当日天气

      Args:
           city: 城市名称

      """
      return f'{city} 当天晴朗 '


model_with_tools = model.bind_tools([get_weather], tool_choice="none")

messages = [
      HumanMessage(" 今天北京天气如何？别瞎编 ")
]

response = model_with_tools.invoke(messages)
response.pretty_print()

C:\Users\lenovo\miniconda3\envs\langchain1.2\Lib\site-packages\langgraph\checkpoint\serde\encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


================================== Ai Message ==================================

我不能确认今天北京的实时天气，避免瞎编。建议你查看“中国天气网”、北京市气象局官方发布，或手机自带天气 App 获取实时信息。


 ### auto 值举例

In [9]:
from langchain.tools import tool
from langchain.messages import HumanMessage
from rich import print as rprint


@tool(parse_docstring=True)
def get_weather(city: str) -> str:
      """
      获取当日天气

      Args:
           city: 城市名称

      """
      return f'{city} 当天晴朗 '


model_with_tools = model.bind_tools([get_weather], tool_choice="auto")

messages = [
      HumanMessage(" 今天北京天气如何？别瞎编 ")
]

responses = model_with_tools.invoke(messages)
for response in responses.tool_calls:
      rprint(response)

{'name': 'get_weather', 'args': {'city': '北京'}, 'id': 'call_00_b1QR2p0g5fQt7QzxFTZt0375', 'type': 'tool_call'}

##  **举例 2：** 不需要调用工具的场景

In [5]:

@tool(parse_docstring=True)
def get_weather(city: str) -> str:
      """
      获取当日天气

      Args:
         city: 城市名称
      """
      return f'{city} 当天晴朗 '


model_with_tools = model.bind_tools([get_weather], tool_choice="auto")

messages = [
      HumanMessage(" 你好啊 ")
]

response = model_with_tools.invoke(messages)
response.pretty_print()

================================== Ai Message ==================================

你好呀！😊 很高兴见到你。

有什么我可以帮你的吗？比如：
- 查一下今天某座城市的天气 ☀️
- 随便聊聊天
- 其他任何问题

随时跟我说～


## required 值举例

In [15]:
from langchain.tools import tool
from langchain.messages import HumanMessage


@tool(parse_docstring=True)
def get_weather(city: str) -> str:
      """
      获取当日天气

      Args:
          city: 城市名称
      """
      return f'{city} 当天晴朗 '


model_with_tools = model.bind_tools([get_weather], tool_choice="required")

messages = [
      HumanMessage(" 今天杭州天气如何？别瞎编 ")
]

response = model_with_tools.invoke(messages, extra_body={"thinking": {"type": "disabled"}})
response.pretty_print()

================================== Ai Message ==================================
Tool Calls:
  get_weather (call_00_v6VMVmb9qyR4WzxPGrvz6868)
 Call ID: call_00_v6VMVmb9qyR4WzxPGrvz6868
  Args:
    city: 杭州


## **举例 2：** 不需要调用工具的场景

In [16]:
from langchain.tools import tool
from langchain.messages import HumanMessage


@tool(parse_docstring=True)
def get_weather(city: str) -> str:
      """
      获取当日天气

      Args:
      city: 城市名称
      """
      f'{city} 当天晴朗 '


model_with_tools = model.bind_tools([get_weather], tool_choice="required")

messages = [
      HumanMessage(" 你好啊 ")
]

response = model_with_tools.invoke(messages,extra_body={"thinking": {"type": "disabled"}})
response.pretty_print()

================================== Ai Message ==================================
Tool Calls:
  get_weather (call_00_YDEA0KvCa7SeXTg96Zey0134)
 Call ID: call_00_YDEA0KvCa7SeXTg96Zey0134
  Args:
    city: 北京
